# 01d -- dim_nfl_teams Seed

**Purpose**: Build `dim_nfl_teams` from nflverse via `nflreadpy.load_teams()`.
Stores canonical NFL team metadata: abbreviations, colors, logos, conference/division.
Used as a join key from `dim_nfl_players` and stat tables.

**Key**: `team_abbr`

**Outputs**:
- `data/dim_nfl_teams.parquet`

In [ ]:
# ---- Setup & Config (shared module) ----------------------------------------
# All config + path anchoring + helpers live in notebooks/etl_helpers.py.
# CFG.data_dir / DATA / REVIEW are anchored to the repo root -> writes always
# land in <repo>/data no matter the CWD this notebook runs from.
import sys
from pathlib import Path
for _p in (Path.cwd() / "notebooks", Path.cwd(), Path.cwd().parent):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import (LeagueConfig, CFG, DATA, REVIEW, TODAY, ALIAS,
                         clean_player_name, generate_player_key, parse_height_to_inches)

import pandas as pd
import numpy as np
import nflreadpy as nfl
from pathlib import Path
from datetime import datetime, date
from dataclasses import dataclass, field


## 1 -- Load Raw Teams

In [ ]:
# -- Load teams from nflreadpy --------------------------------------------------
print("Loading NFL teams from nflverse...")
try:
    teams_raw = nfl.load_teams().to_pandas()
except Exception as e:
    raise RuntimeError(f"Failed to load nflverse teams: {e}") from e

print(f"Raw rows: {len(teams_raw)}")
print(f"Columns: {list(teams_raw.columns)}")

Loading NFL teams from nflverse...
Raw rows: 36
Columns: ['team_abbr', 'team_name', 'team_id', 'team_nick', 'team_conf', 'team_division', 'team_color', 'team_color2', 'team_color3', 'team_color4', 'team_logo_wikipedia', 'team_logo_espn', 'team_wordmark', 'team_conference_logo', 'team_league_logo', 'team_logo_squared']


## 2 -- Select Columns and Save

In [ ]:
# -- Select and store key columns -----------------------------------------------
# Drop redundant logo variants; keep the columns used for joins and display.
_KEEP = [
    "team_abbr",           # primary key
    "team_name",
    "team_id_pfr",
    "team_conf",
    "team_division",
    "team_color",
    "team_color2",
    "team_logo_wikipedia",
    "team_logo_espn",
    "team_wordmark",
    "team_stadium",
]

_missing   = [c for c in _KEEP if c not in teams_raw.columns]
if _missing:
    print(f"Note: columns not in this nflverse build (kept as NA): {_missing}")

# reindex selects existing columns, adds missing as NA, and orders — one call.
dim_nfl_teams = teams_raw.reindex(columns=_KEEP).copy()

# Drop rows with no team_abbr (nflverse sometimes includes summary/empty rows)
dim_nfl_teams = dim_nfl_teams[dim_nfl_teams["team_abbr"].notna()].reset_index(drop=True)

out_path = DATA / "dim_nfl_teams.parquet"
dim_nfl_teams.to_parquet(out_path, index=False)
print(f"dim_nfl_teams: {len(dim_nfl_teams)} teams -> {out_path}")
dim_nfl_teams.head(8)

Note: columns not in this nflverse build (kept as NA): ['team_id_pfr', 'team_stadium']
dim_nfl_teams: 36 teams -> data\dim_nfl_teams.parquet


,team_abbr,team_name,team_id_pfr,team_conf,team_division,team_color,team_color2,team_logo_wikipedia,team_logo_espn,team_wordmark,team_stadium
0,ARI,Arizona Cardinals,<NA>,NFC,NFC West,#97233F,#000000,https://upload.wikimedia.org/wikipedia/en/thum...,https://a.espncdn.com/i/teamlogos/nfl/500/ari.png,https://github.com/nflverse/nflverse-pbp/raw/m...,<NA>
1,ATL,Atlanta Falcons,<NA>,NFC,NFC South,#A71930,#000000,https://upload.wikimedia.org/wikipedia/en/thum...,https://a.espncdn.com/i/teamlogos/nfl/500/atl.png,https://github.com/nflverse/nflverse-pbp/raw/m...,<NA>
2,BAL,Baltimore Ravens,<NA>,AFC,AFC North,#241773,#9E7C0C,https://upload.wikimedia.org/wikipedia/en/thum...,https://a.espncdn.com/i/teamlogos/nfl/500/bal.png,https://github.com/nflverse/nflverse-pbp/raw/m...,<NA>
3,BUF,Buffalo Bills,<NA>,AFC,AFC East,#00338D,#C60C30,https://upload.wikimedia.org/wikipedia/en/thum...,https://a.espncdn.com/i/teamlogos/nfl/500/buf.png,https://github.com/nflverse/nflverse-pbp/raw/m...,<NA>
4,CAR,Carolina Panthers,<NA>,NFC,NFC South,#0085CA,#000000,https://upload.wikimedia.org/wikipedia/en/thum...,https://a.espncdn.com/i/teamlogos/nfl/500-dark...,https://github.com/nflverse/nflverse-pbp/raw/m...,<NA>
5,CHI,Chicago Bears,<NA>,NFC,NFC North,#0B162A,#E64100,https://upload.wikimedia.org/wikipedia/commons...,https://a.espncdn.com/i/teamlogos/nfl/500/chi.png,https://github.com/nflverse/nflverse-pbp/raw/m...,<NA>
6,CIN,Cincinnati Bengals,<NA>,AFC,AFC North,#FB4F14,#000000,https://upload.wikimedia.org/wikipedia/commons...,https://a.espncdn.com/i/teamlogos/nfl/500/cin.png,https://github.com/nflverse/nflverse-pbp/raw/m...,<NA>
7,CLE,Cleveland Browns,<NA>,AFC,AFC North,#FF3C00,#311D00,https://upload.wikimedia.org/wikipedia/en/thum...,https://a.espncdn.com/i/teamlogos/nfl/500/cle.png,https://github.com/nflverse/nflverse-pbp/raw/m...,<NA>


## 3 -- Validation

In [ ]:
df = pd.read_parquet(DATA / "dim_nfl_teams.parquet")
print(f"dim_nfl_teams: {len(df)} rows, {len(df.columns)} columns")
print()
print("By conference / division:")
print(df.groupby(["team_conf", "team_division"])["team_abbr"].apply(list).to_string())

dim_nfl_teams: 36 rows, 11 columns

By conference / division:
team_conf  team_division
AFC        AFC East                  [BUF, MIA, NE, NYJ]
           AFC North                [BAL, CIN, CLE, PIT]
           AFC South                [HOU, IND, JAX, TEN]
           AFC West          [DEN, KC, LAC, LV, OAK, SD]
NFC        NFC East                 [DAL, NYG, PHI, WAS]
           NFC North                 [CHI, DET, GB, MIN]
           NFC South                  [ATL, CAR, NO, TB]
           NFC West         [ARI, LA, LAR, SEA, SF, STL]
